# RQ2 Notebook 02 — Master Pairwise Dataset

Builds the master pairwise table that every later RQ2 notebook reads.

**Unit of analysis:** `(CVE, upstream G:A, downstream G:A)`

**Target:** 1,677 pairs / 191 CVEs / 742 downstream repos, 7 lifecycle patterns

Each step prints a DataFrame so the filtering is auditable. Read the markdown above each
step for *why* rows are removed.

Produces `T9` (construction chain) and `T10` (selection-bias check).

In [89]:
import contextlib
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

DATA = Path('../../data/baseline')
ELEC = Path('../../data/rq1/external_release_nvd')
RQ1  = Path('../../data/rq1')
SURV = Path('../../data/rq1')
PROG = Path('../../data/rq1')
OUT  = Path('../../data/rq2')

RESPONSE_CSV   = DATA / 'RESPONSE_with_commit_dates.csv'
PATCH_CSV      = DATA / 'PATCH.csv'
DEP_CSV        = DATA / 'DEP.csv'
CVE_CSV        = DATA / 'CVE.csv'
ARTIFACT_CSV   = DATA / 'ARTIFACT.csv'
FIX_RELEASE    = ELEC / 'fix_releases_from_patch_data_local_server.csv'
NVD_CSV        = ELEC / 'cve_NVD_disclosure_dates.csv'
LINKS_CSV      = RQ1  / 'corrected_resolved_links_v2.csv'
SEVERITY_CSV   = SURV / 'repo_with_severity.csv'
POPULARITY_CSV = PROG / 'repo_popularity_at_fix_commit.csv'
ADJUSTED_NB    = PROG / 'transparent_data' / 'rq1_transparent_lifecycle_order_adjusted.ipynb'

# Rows are tracked here so the construction chain (T9) is built from the real numbers.
chain = []

def log_step(step, df, note, cve_col='CVE', repo_col='repo'):
    chain.append({
        'Step': step,
        'Rows': len(df),
        'Unique CVEs': df[cve_col].nunique() if cve_col in df.columns else np.nan,
        'Unique downstream repos': df[repo_col].nunique() if repo_col in df.columns else np.nan,
        'Note': note,
    })
    return df

print('paths configured')

paths configured


---
## Step 1 — Load the mined mitigation commits

`RESPONSE_with_commit_dates.csv` is the output of notebook 01. It is `RESPONSE.csv` plus a
mined `COMMIT_DATE` per commit URL.

Nothing is filtered here. This is the raw starting point.

In [90]:
raw = pd.read_csv(RESPONSE_CSV)

response = raw[['CVE', 'UPSTREAM G:A:V', 'DOWNSTREAM G:A:V', 'DOWNSTREAM REPO',
                'COMMIT', 'COMMIT_DATE', 'COMMIT_DATE_ERROR']].copy()
response.columns = ['CVE', 'upstream_GA', 'downstream_GA', 'downstream_repo',
                    'commit_url', 'commit_date_raw', 'commit_date_error']

log_step('0. RESPONSE_with_commit_dates.csv (raw)', response,
         'Starting point. One row per mined mitigation record.',
         repo_col='downstream_repo')

print(f'rows: {len(response):,}')
print(f'unique CVEs: {response.CVE.nunique()}')
print(f'unique downstream repos: {response.downstream_repo.nunique()}')
print(f'unique commit URLs: {response.commit_url.nunique():,}')
response.head(5)

rows: 3,312
unique CVEs: 240
unique downstream repos: 987
unique commit URLs: 1,608


,CVE,upstream_GA,downstream_GA,downstream_repo,commit_url,commit_date_raw,commit_date_error
0,CVE-2021-39154,com.thoughtworks.xstream:xstream,ne,dbmdz/digitalcollections-model,https://github.com/dbmdz/digitalcollections-mo...,2020-11-18T08:28:46Z,NaN
1,CVE-2021-37714,org.jsoup:jsoup,com.github.btheu.estivate:estivate,btheu/estivate,https://github.com/btheu/estivate/commit/22adf...,2021-12-03T15:44:15Z,NaN
2,CVE-2021-37714,org.jsoup:jsoup,com.jcabi:jcabi-http,jcabi/jcabi-http,https://github.com/jcabi/jcabi-http/commit/1a2...,2021-08-16T23:00:43Z,NaN
3,CVE-2021-37714,org.jsoup:jsoup,in.ashwanthkumar:gocd-java-client,ashwanthkumar/gocd-java-client,https://github.com/ashwanthkumar/gocd-java-cli...,2021-09-30T23:03:50Z,NaN
4,CVE-2021-37714,org.jsoup:jsoup,tech.grasshopper:pdfextentreporter,grasshopper7/pdfextentreporter,https://github.com/grasshopper7/pdfextentrepor...,2022-03-01T12:31:07Z,NaN


---
## Step 2 — Remove duplicate rows

**Why:** `RESPONSE.csv` contains byte-identical repeated rows. These are not multiple
mitigation events — they are the same record emitted more than once by the upstream mining.

**Verified property:** every `(CVE, upstream G:A, downstream G:A)` triple maps to exactly
one commit and one repo. So there is no ambiguity about which commit counts as *the*
mitigation, and no tie-breaking rule is needed. A plain `drop_duplicates()` is correct.

The check below confirms that property on your data rather than assuming it.

In [91]:
key = ['CVE', 'upstream_GA', 'downstream_GA']

# Confirm the one-commit-per-triple property before deduplicating.
grp = response.groupby(key)
property_check = pd.DataFrame([{
    'Distinct triples': grp.ngroups,
    'Triples with >1 row': int((grp.size() > 1).sum()),
    'Triples with >1 distinct commit': int((grp.commit_url.nunique() > 1).sum()),
    'Triples with >1 distinct repo': int((grp.downstream_repo.nunique() > 1).sum()),
}])
display(property_check)

n_before = len(response)
response = response.drop_duplicates().reset_index(drop=True)

dedup_table = pd.DataFrame([
    {'Stage': 'Before dedup', 'Rows': n_before},
    {'Stage': 'After dedup',  'Rows': len(response)},
    {'Stage': 'Removed (exact duplicates)', 'Rows': n_before - len(response)},
])
display(dedup_table)

log_step('1. drop_duplicates()', response,
         'Removed byte-identical repeated rows. Not distinct mitigation events.',
         repo_col='downstream_repo')

assert property_check['Triples with >1 distinct commit'].iloc[0] == 0, \
    'Multiple commits per triple: a tie-breaking rule IS needed. Stop and revisit.'
print('OK: one commit per triple confirmed. No tie-breaking rule required.')

,Distinct triples,Triples with >1 row,Triples with >1 distinct commit,Triples with >1 distinct repo
0,2269,594,0,0


,Stage,Rows
0,Before dedup,3312
1,After dedup,2269
2,Removed (exact duplicates),1043


OK: one commit per triple confirmed. No tie-breaking rule required.


---
## Step 3 — Drop malformed downstream identifiers

**Why:** the `DOWNSTREAM G:A:V` column is misnamed — it holds `G:A` values, not versions —
and contains at least one junk entry. A downstream identifier that does not resolve to a
real artifact cannot be joined to `DEP.csv` or `ARTIFACT.csv`, so it cannot participate in
any exposure calculation.

We identify these by checking which downstream values have no match in `DEP.csv`.

In [92]:
dep = pd.read_csv(DEP_CSV)
dep['dep_downstream_GA'] = dep['Downstream G:A:V'].str.rsplit(':', n=1).str[0]
valid_downstream = set(dep['dep_downstream_GA'])

response['downstream_in_DEP'] = response.downstream_GA.isin(valid_downstream)

orphans = response[~response.downstream_in_DEP]
display(orphans[['CVE', 'upstream_GA', 'downstream_GA', 'downstream_repo']])

print(f'orphan rows (no match in DEP.csv): {len(orphans)}')

response = response[response.downstream_in_DEP].drop(columns=['downstream_in_DEP']).reset_index(drop=True)

log_step('2. Drop downstream IDs absent from DEP.csv', response,
         'Malformed downstream identifiers. Cannot be joined to exposure data.',
         repo_col='downstream_repo')

print(f'rows remaining: {len(response):,}')

,CVE,upstream_GA,downstream_GA,downstream_repo
0,CVE-2021-39154,com.thoughtworks.xstream:xstream,ne,dbmdz/digitalcollections-model


orphan rows (no match in DEP.csv): 1
rows remaining: 2,268


---
## Step 4 — Parse mined commit dates

**Why:** notebook 01 could not resolve every commit URL. All failures returned
`404_not_found`, meaning the downstream repository was deleted or renamed, or the commit
was removed by a force-push.

**We do NOT drop these yet.** They are kept through the RQ1 join so we can test whether
missing dates concentrate in one class (Step 8). If they do, the analysis set is biased and
that must be reported rather than silently absorbed.

In [93]:
response['adoption_date'] = pd.to_datetime(
    response.commit_date_raw, errors='coerce', utc=True
).dt.tz_convert(None)

response['has_date'] = response.adoption_date.notna()

date_status = (
    response.commit_date_error.fillna('no_error')
    .value_counts()
    .rename_axis('Mining status')
    .reset_index(name='Pairs')
)
date_status['%'] = (100 * date_status.Pairs / len(response)).round(2)
display(date_status)

print(f'pairs with a usable date : {response.has_date.sum():,}')
print(f'pairs missing a date     : {(~response.has_date).sum():,}')
print(f'date range               : {response.adoption_date.min().date()} to {response.adoption_date.max().date()}')

,Mining status,Pairs,%
0,no_error,2227,98.19
1,404_not_found,41,1.81


pairs with a usable date : 2,227
pairs missing a date     : 41
date range               : 2010-06-16 to 2022-08-03


---
## Step 5 — Rebuild the RQ1 silent lifecycle patterns

**Why:** RQ2 needs each CVE's lifecycle pattern and event dates. The silent side is rebuilt
here using exactly the same rules as RQ1 notebook 05, so the two research questions stay
consistent.

Two filters are inherited from RQ1:

1. `Oldest Tag Date == 'not found'` — no release date could be mined, so the lifecycle is
   incomplete.
2. `Release Date < Fix Date` — a tag predating its own fix commit is a tag-selection
   artifact, not a real ordering.

Both are RQ1's filters, not RQ2's. CVEs removed here were never in RQ1's lifecycle analysis
either.

In [94]:
fix_release = pd.read_csv(FIX_RELEASE)
nvd         = pd.read_csv(NVD_CSV)
links       = pd.read_csv(LINKS_CSV)

n_all = len(fix_release)

lifecycle = fix_release[
    fix_release['Oldest Tag Date'].astype(str).str.lower().ne('not found')
].copy()
n_has_release = len(lifecycle)

lifecycle['fix_date']     = pd.to_datetime(lifecycle['Commit Date'], errors='coerce')
lifecycle['release_date'] = pd.to_datetime(lifecycle['Oldest Tag Date'], errors='coerce')
lifecycle = lifecycle.merge(nvd[['CVE_ID', 'Published Date']], on='CVE_ID', how='left')
lifecycle['disclosure_date'] = pd.to_datetime(lifecycle['Published Date'], errors='coerce')
lifecycle = lifecycle.merge(links[['CVE_ID', 'Link Presence']], on='CVE_ID', how='left')
lifecycle['rq1_reporting'] = lifecycle['Link Presence'].map(
    {'contains links': 'Transparent', 'no links': 'Silent'}
)

lifecycle = lifecycle.dropna(subset=['fix_date', 'release_date', 'disclosure_date', 'rq1_reporting'])
n_complete = len(lifecycle)

lifecycle = lifecycle[lifecycle.release_date >= lifecycle.fix_date]
n_ordered = len(lifecycle)

rq1_filter_table = pd.DataFrame([
    {'RQ1 filter': 'All CVEs in fix_releases', 'CVEs': n_all, 'Removed': 0},
    {'RQ1 filter': "Oldest Tag Date != 'not found'", 'CVEs': n_has_release, 'Removed': n_all - n_has_release},
    {'RQ1 filter': 'Fix/Release/Disclosure all parse', 'CVEs': n_complete, 'Removed': n_has_release - n_complete},
    {'RQ1 filter': 'Release Date >= Fix Date', 'CVEs': n_ordered, 'Removed': n_complete - n_ordered},
])
display(rq1_filter_table)

def order_three(row):
    events = [('Fix', row.fix_date), ('Release', row.release_date), ('Disclosure', row.disclosure_date)]
    tie = {'Fix': 0, 'Release': 1, 'Disclosure': 2}
    return tuple(name for name, _ in sorted(events, key=lambda e: (e[1], tie[e[0]])))

silent = lifecycle[lifecycle.rq1_reporting.eq('Silent')].copy()
silent['pattern'] = silent.apply(order_three, axis=1).map({
    ('Fix', 'Release', 'Disclosure'): 'S1',
    ('Fix', 'Disclosure', 'Release'): 'S2',
    ('Disclosure', 'Fix', 'Release'): 'S3',
})
silent['report_date'] = pd.NaT
silent['original_pattern'] = silent['pattern']
silent['gad_reclassified'] = False
silent = silent[['CVE_ID', 'pattern', 'original_pattern', 'gad_reclassified',
                 'report_date', 'fix_date', 'release_date', 'disclosure_date']]

display(
    silent.pattern.value_counts().rename_axis('Silent pattern').reset_index(name='CVEs')
)
print(f'silent lifecycle CVEs: {len(silent)}')

,RQ1 filter,CVEs,Removed
0,All CVEs in fix_releases,832,0
1,Oldest Tag Date != 'not found',758,74
2,Fix/Release/Disclosure all parse,758,0
3,Release Date >= Fix Date,743,15


,Silent pattern,CVEs
0,S1,258
1,S2,38
2,S3,19


silent lifecycle CVEs: 315


---
## Step 6 — Load the adjusted transparent lifecycle

**Why:** the transparent side carries manual corrections that live in
`rq1_transparent_lifecycle_order_adjusted.ipynb`. Re-implementing them here would risk
drift, so that notebook is executed in memory and its final DataFrame is read directly.

That notebook already applied its own filters: report dates required, missing release dates
removed, `Release < Fix` removed, and one manual exclusion (`CVE-2013-5679`). Its output is
278 CVEs.

In [95]:
adjusted_nb = json.loads(ADJUSTED_NB.read_text())
adjusted_ns = {}

with contextlib.redirect_stdout(io.StringIO()):
    for i, cell in enumerate(adjusted_nb['cells']):
        if cell.get('cell_type') == 'code':
            source = ''.join(cell.get('source', []))
            exec(compile(source, f'adjusted_cell_{i}', 'exec'), adjusted_ns)

transparent_raw = adjusted_ns['transparent_lifecycle_order_adjusted_df'].copy()
print(f'adjusted transparent lifecycle rows: {len(transparent_raw)}')

ORIGINAL_PATTERN_KEY = {
    ('Report', 'Fix', 'Release', 'Disclosure'): 'T1',
    ('Report', 'Fix', 'Disclosure', 'Release'): 'T2',
    ('Disclosure', 'Report', 'Fix', 'Release'): 'T3',
    ('Fix', 'Release', 'Disclosure', 'GAD'):    'T4',
    ('Report', 'Disclosure', 'Fix', 'Release'): 'T5',
    ('Disclosure', 'Fix', 'Release', 'GAD'):    'T6',
    ('Fix', 'Disclosure', 'Release', 'GAD'):    'T7',
}

# RQ1 notebook 09: GAD-ending patterns are reclassified as silent-equivalent, because the
# link points at a vulnerability-database entry rather than an issue/PR/report workflow.
GAD_TO_SILENT = {'T4': 'S1', 'T6': 'S3', 'T7': 'S2'}

transparent_raw['original_pattern'] = transparent_raw.apply(
    lambda r: ORIGINAL_PATTERN_KEY[(r['First'], r['Second'], r['Third'], r['Fourth'])], axis=1
)
transparent_raw['pattern'] = transparent_raw.original_pattern.map(lambda p: GAD_TO_SILENT.get(p, p))
transparent_raw['gad_reclassified'] = transparent_raw.original_pattern.isin(GAD_TO_SILENT)

transparent = transparent_raw.rename(columns={
    'Report Date': 'report_date',
    'Fix Date': 'fix_date',
    'Release Date': 'release_date',
    'Disclosure Date': 'disclosure_date',
})[['CVE_ID', 'pattern', 'original_pattern', 'gad_reclassified',
    'report_date', 'fix_date', 'release_date', 'disclosure_date']]

for c in ['report_date', 'fix_date', 'release_date', 'disclosure_date']:
    transparent[c] = pd.to_datetime(transparent[c], errors='coerce')

# A GAD entry is not a report. These rows carry no report, exactly like any other silent fix.
transparent.loc[transparent.gad_reclassified, 'report_date'] = pd.NaT

display(
    transparent.groupby(['original_pattern', 'pattern', 'gad_reclassified'])
    .size().rename('CVEs').reset_index().sort_values('original_pattern')
)
print(f'GAD-ending CVEs reclassified to silent: {int(transparent.gad_reclassified.sum())}')
print('  report_date cleared for these rows (no report, as with any silent fix)')

adjusted transparent lifecycle rows: 278


,original_pattern,pattern,gad_reclassified,CVEs
0,T1,T1,False,216
1,T2,T2,False,26
2,T3,T3,False,13
3,T4,S1,True,13
4,T5,T5,False,6
5,T6,S3,True,2
6,T7,S2,True,2


GAD-ending CVEs reclassified to silent: 17
  report_date cleared for these rows (no report, as with any silent fix)


---
## Step 7 — Reclassify using the report-before-fix rule

**Why:** RQ1 labelled a fix "transparent" when the commit message contained a link. That is
a presence test, not a timing test. Patterns `T4`, `T6`, and `T7` end in `GAD`, a link to a
vulnerability database rather than an issue/PR/report workflow, and it appears *after* the
fix, release, and disclosure. Nothing was reported before the developer fixed the bug.

**RQ1 notebook 09** therefore merges these into their silent equivalents:

| Original | Becomes |
|---|---|
| `T4: Fix → Release → Disclosure → GAD` | `S1: Fix → Release → Disclosure → None` |
| `T6: Disclosure → Fix → Release → GAD` | `S3: Disclosure → Fix → Release → None` |
| `T7: Fix → Disclosure → Release → GAD` | `S2: Fix → Disclosure → Release → None` |

RQ2 adopts this and arrives at the same classes by an independent route:

> **Transparent = a report exists AND precedes the fix commit.**

All four surviving transparent patterns (T1, T2, T3, T5) have Report before Fix; the three
silent patterns have no pre-fix report. Two different justifications, identical assignment.

`original_pattern` is retained on every row for traceability. Reclassified rows carry no
report date — a GAD entry is a database record, not a report.

In [96]:
SILENT_PATTERNS = {'S1', 'S2', 'S3'}

rq1_lifecycle = pd.concat([silent, transparent], ignore_index=True)
rq1_lifecycle['class'] = np.where(
    rq1_lifecycle.pattern.isin(SILENT_PATTERNS), 'Silent', 'Transparent'
)

# Verify the rule holds: transparent rows must have report_date < fix_date.
tr = rq1_lifecycle[rq1_lifecycle['class'].eq('Transparent')]
violations = int((tr.report_date >= tr.fix_date).sum())

reclass_table = (
    rq1_lifecycle.groupby(['class', 'pattern', 'original_pattern']).size()
    .rename('CVEs').reset_index()
    .sort_values(['class', 'pattern', 'original_pattern'])
)
display(reclass_table)

display(
    rq1_lifecycle.groupby('class').size().rename('CVEs').reset_index()
)

print(f'total RQ1 lifecycle CVEs: {len(rq1_lifecycle)}')
print(f'transparent rows where report_date >= fix_date: {violations}')
print('(a small number is expected from same-timestamp ties; a large number means the rule is wrong)')

,class,pattern,original_pattern,CVEs
0,Silent,S1,S1,258
1,Silent,S1,T4,13
2,Silent,S2,S2,38
3,Silent,S2,T7,2
4,Silent,S3,S3,19
5,Silent,S3,T6,2
6,Transparent,T1,T1,216
7,Transparent,T2,T2,26
8,Transparent,T3,T3,13
9,Transparent,T5,T5,6


,class,CVEs
0,Silent,332
1,Transparent,261


total RQ1 lifecycle CVEs: 593
transparent rows where report_date >= fix_date: 31
(a small number is expected from same-timestamp ties; a large number means the rule is wrong)


---
## Step 8 — Join mitigations to RQ1 lifecycle, then test the missing dates

**Why this order matters:** the join happens *before* dropping undated pairs so we can test
whether missing dates are randomly distributed across classes.

- If missing dates are balanced across Silent/Transparent, dropping them is harmless.
- If they concentrate in one class, the analysis set is biased and `T10` must say so.

The join also drops mitigation pairs whose CVE has no complete RQ1 lifecycle. Those CVEs
were removed by RQ1's filters (Step 5), not by RQ2.

In [97]:
joined_all = response.merge(
    rq1_lifecycle, left_on='CVE', right_on='CVE_ID', how='inner'
).drop(columns=['CVE_ID'])

dropped_no_lifecycle = response[~response.CVE.isin(set(rq1_lifecycle.CVE_ID))]

join_table = pd.DataFrame([
    {'Set': 'Mitigation pairs entering join', 'Pairs': len(response), 'CVEs': response.CVE.nunique()},
    {'Set': 'Matched to RQ1 lifecycle', 'Pairs': len(joined_all), 'CVEs': joined_all.CVE.nunique()},
    {'Set': 'Dropped: CVE has no complete RQ1 lifecycle',
     'Pairs': len(dropped_no_lifecycle), 'CVEs': dropped_no_lifecycle.CVE.nunique()},
])
display(join_table)

log_step('3. Join to RQ1 lifecycle', joined_all,
         'Dropped pairs whose CVE lacks a complete RQ1 lifecycle (RQ1 filters, not RQ2).',
         repo_col='downstream_repo')

# --- Missing-date bias test -------------------------------------------------
miss_tab = pd.crosstab(joined_all['class'], joined_all['has_date'])
miss_tab.columns = ['Missing date', 'Has date']
miss_tab['% missing'] = (100 * miss_tab['Missing date'] /
                         (miss_tab['Missing date'] + miss_tab['Has date'])).round(2)
display(miss_tab)

if miss_tab['Missing date'].sum() > 0 and miss_tab.shape[0] > 1:
    chi2, p, _, _ = chi2_contingency(miss_tab[['Missing date', 'Has date']].values)
    print(f'chi-square test, missing date vs class: chi2={chi2:.3f}, p={p:.4f}')
    if p < 0.05:
        print('WARNING: missing dates are NOT balanced across classes. Report this in T10.')
    else:
        print('OK: missing dates are balanced across classes. Dropping them is defensible.')

,Set,Pairs,CVEs
0,Mitigation pairs entering join,2268,239
1,Matched to RQ1 lifecycle,1829,195
2,Dropped: CVE has no complete RQ1 lifecycle,439,44


,Missing date,Has date,% missing
class,,,
Silent,17,1183,1.42
Transparent,15,614,2.38


chi-square test, missing date vs class: chi2=1.722, p=0.1894
OK: missing dates are balanced across classes. Dropping them is defensible.


---
## Step 9 — Drop undated pairs

**Why:** every timing, stage, and burden calculation requires an adoption date. Pairs
without one cannot contribute and are removed here — after the bias test above, not before.

In [98]:
master = joined_all[joined_all.has_date].drop(columns=['has_date']).reset_index(drop=True)

drop_table = pd.DataFrame([
    {'Stage': 'Joined pairs (dated + undated)', 'Pairs': len(joined_all)},
    {'Stage': 'Dropped: no mined commit date', 'Pairs': len(joined_all) - len(master)},
    {'Stage': 'FINAL ANALYSIS SET', 'Pairs': len(master)},
])
display(drop_table)

log_step('4. Drop pairs with no mined commit date', master,
         'Timing analysis requires an adoption date. All failures were 404_not_found.',
         repo_col='downstream_repo')

print(f'pairs : {len(master):,}')
print(f'CVEs  : {master.CVE.nunique()}')
print(f'repos : {master.downstream_repo.nunique()}')

,Stage,Pairs
0,Joined pairs (dated + undated),1829
1,Dropped: no mined commit date,32
2,FINAL ANALYSIS SET,1797


pairs : 1,797
CVEs  : 195
repos : 779


---
## Step 10 — Compute anchored delays

**Why signed, not absolute:** a negative delay means the downstream project mitigated
*before* that upstream event. That is informative — for example, a project tracking
snapshot builds may adopt a fix before the official release tag exists. Clamping to zero
would hide it.

```
delay_fix        = adoption - fix
delay_release    = adoption - release
delay_disclosure = adoption - disclosure
delay_report     = adoption - report        (transparent only; NaN for silent)
```

No composite "exposure start" is invented. Each anchor is reported on its own terms.

In [99]:
for anchor in ['fix', 'release', 'disclosure', 'report']:
    master[f'delay_{anchor}'] = (
        master.adoption_date - master[f'{anchor}_date']
    ).dt.total_seconds() / 86400

delay_summary = []
for anchor in ['fix', 'release', 'disclosure', 'report']:
    v = master[f'delay_{anchor}'].dropna()
    delay_summary.append({
        'Anchor': anchor,
        'Pairs with anchor': len(v),
        'Median days': round(v.median(), 1),
        'Q1': round(v.quantile(0.25), 1),
        'Q3': round(v.quantile(0.75), 1),
        'Adopted AFTER anchor (n)': int((v > 0).sum()),
        'Adopted AFTER anchor (%)': round(100 * (v > 0).mean(), 1),
    })

delay_table = pd.DataFrame(delay_summary)
display(delay_table)

print('Benchmark from RQ1: the S1 release-to-disclosure window is 29.86 days (median).')
print('Compare against median delay_release above.')

,Anchor,Pairs with anchor,Median days,Q1,Q3,Adopted AFTER anchor (n),Adopted AFTER anchor (%)
0,fix,1797,105.3,34.4,337.7,1677,93.3
1,release,1797,71.8,9.1,308.1,1615,89.9
2,disclosure,1797,48.3,2.4,210.3,1442,80.2
3,report,614,119.6,32.8,336.7,581,94.6


Benchmark from RQ1: the S1 release-to-disclosure window is 29.86 days (median).
Compare against median delay_release above.


---
## Step 10b — Exclude adoptions that predate the upstream fix

**The rule:** `delay_fix >= 0`.

A downstream commit dated **before** the upstream security fix commit cannot be an adoption
of that fix — the fix did not exist yet. Whatever those commits were (a dependency bump for
an unrelated reason, a version pin, a coincidental match by the mining), they did not take
up a patch that had not been written.

**What this does NOT remove.** The filter compares two dates and nothing else. Disclosure,
release, and report timing are never consulted. In particular, the sequence *disclosed first
-> upstream fixes later -> downstream adopts* is fully retained: an adoption 873 days after
a late upstream fix is kept, because the fix existed when the commit was made.

**Expected:** 120 pairs removed (6.7%), median 54 days early, range 1 to 596 days. Spread
across 64 CVEs and 91 repos, so this is per-commit mis-attribution rather than one bad CVE.
Removal is near-balanced by class (7.0% silent, 6.0% transparent), so it does not tilt the
RQ2.4 comparison.

In [100]:
before_n = len(master)
excluded_prefix = master[master.delay_fix < 0].copy()
master = master[master.delay_fix >= 0].reset_index(drop=True)

exclusion_table = pd.DataFrame([
    {'Stage': 'Before exclusion', 'Pairs': before_n},
    {'Stage': 'Excluded (adoption predates upstream fix)', 'Pairs': len(excluded_prefix)},
    {'Stage': 'After exclusion', 'Pairs': len(master)},
])
display(exclusion_table)

early_days = -excluded_prefix.delay_fix
bands = [('1-7 days', (early_days >= 1) & (early_days < 7)),
         ('7-30 days', (early_days >= 7) & (early_days < 30)),
         ('30-90 days', (early_days >= 30) & (early_days < 90)),
         ('90-365 days', (early_days >= 90) & (early_days < 365)),
         ('>365 days', early_days >= 365)]
display(pd.DataFrame([{'How early': n, 'Pairs': int(m_.sum()),
                       '%': round(100 * m_.mean(), 1)} for n, m_ in bands]))

display(
    excluded_prefix.groupby(['class', 'pattern']).size().rename('Excluded').reset_index()
    .merge(joined_all.groupby(['class', 'pattern']).size().rename('Total').reset_index(),
           on=['class', 'pattern'])
    .assign(**{'% of pattern': lambda d: (100 * d.Excluded / d.Total).round(1)})
)

log_step('5. Exclude adoption before upstream fix', master,
         'delay_fix < 0. A commit cannot adopt a fix that does not yet exist.',
         repo_col='downstream_repo')

print(f'excluded : {len(excluded_prefix)} pairs '
      f'({excluded_prefix.CVE.nunique()} CVEs, {excluded_prefix.downstream_repo.nunique()} repos affected)')
print(f'remaining: {len(master):,} pairs | {master.CVE.nunique()} CVEs | '
      f'{master.downstream_repo.nunique()} repos')
print()
print('CVEs lost entirely:', sorted(set(excluded_prefix.CVE) - set(master.CVE)))

,Stage,Pairs
0,Before exclusion,1797
1,Excluded (adoption predates upstream fix),120
2,After exclusion,1677


,How early,Pairs,%
0,1-7 days,5,4.2
1,7-30 days,35,29.2
2,30-90 days,42,35.0
3,90-365 days,33,27.5
4,>365 days,5,4.2


,class,pattern,Excluded,Total,% of pattern
0,Silent,S1,80,1108,7.2
1,Silent,S2,2,70,2.9
2,Silent,S3,1,22,4.5
3,Transparent,T1,20,552,3.6
4,Transparent,T2,2,38,5.3
5,Transparent,T3,14,37,37.8
6,Transparent,T5,1,2,50.0


excluded : 120 pairs (64 CVEs, 91 repos affected)
remaining: 1,677 pairs | 191 CVEs | 742 repos

CVEs lost entirely: ['CVE-2017-9799', 'CVE-2018-8010', 'CVE-2018-8026', 'CVE-2019-10241']


---
## Step 11 — Classify each mitigation into a lifecycle stage

**Why:** RQ2.2 asks *where* in the upstream lifecycle downstream exposure ends. Each
mitigation is placed in the interval between consecutive upstream events.

The event list is built per row from the dates that exist, then sorted. This handles every
pattern uniformly — including `T4`/`T6`/`T7`, where the advisory (`GAD`) comes last, and the
silent patterns, which have no report at all.

In [101]:
def mitigation_stage(row):
    events = []
    if pd.notna(row.report_date):
        events.append(('Report', row.report_date))
    events.append(('Fix', row.fix_date))
    events.append(('Release', row.release_date))
    events.append(('Disclosure', row.disclosure_date))
    events.sort(key=lambda e: e[1])

    a = row.adoption_date
    if a < events[0][1]:
        return f'Before {events[0][0]}'
    for i in range(len(events) - 1):
        if events[i][1] <= a < events[i + 1][1]:
            return f'{events[i][0]}->{events[i + 1][0]}'
    return f'After {events[-1][0]}'


master['mitigation_stage'] = master.apply(mitigation_stage, axis=1)

stage_by_class = pd.crosstab(master['class'], master.mitigation_stage)
display(stage_by_class)

stage_pct = pd.crosstab(master['class'], master.mitigation_stage, normalize='index').mul(100).round(1)
display(stage_pct)

display(
    master.mitigation_stage.value_counts()
    .rename_axis('Mitigation stage').reset_index(name='Pairs')
    .assign(**{'%': lambda d: (100 * d.Pairs / len(master)).round(1)})
)

mitigation_stage,After Disclosure,After Release,Disclosure->Release,Fix->Disclosure,Fix->Release,Release->Disclosure,Report->Disclosure
class,,,,,,,
Silent,894,61,19,0,15,111,0
Transparent,399,39,9,7,12,102,9


mitigation_stage,After Disclosure,After Release,Disclosure->Release,Fix->Disclosure,Fix->Release,Release->Disclosure,Report->Disclosure
class,,,,,,,
Silent,81.3,5.5,1.7,0.0,1.4,10.1,0.0
Transparent,69.2,6.8,1.6,1.2,2.1,17.7,1.6


,Mitigation stage,Pairs,%
0,After Disclosure,1293,77.1
1,Release->Disclosure,213,12.7
2,After Release,100,6.0
3,Disclosure->Release,28,1.7
4,Fix->Release,27,1.6
5,Report->Disclosure,9,0.5
6,Fix->Disclosure,7,0.4


---
## Step 12 — Attach covariates

**Why each one:**

| Covariate | Needed for |
|---|---|
| `severity`, `cvss` | Stratification and model control |
| `commit_n_CVEs` | How many distinct CVEs one downstream commit clears. Multi-CVE commits are routine maintenance, not targeted response. Counted from the unfiltered mitigation records so it describes the commit, not our sample. Modelled, never deleted — dropping them would select on a correlate of the treatment |
| `downstream_usage_num`, `downstream_loc` | Downstream project size and activity drive adoption speed independently of upstream behavior |
| `upstream_stars_at_fix` | Upstream popularity confound |
| `dependents` | Blast radius per CVE, needed for burden in notebook 05 |

`ARTIFACT.csv` is version-level, so it is aggregated to `G:A`: max `USAGE_NUM` and median
`LOC`/`CLASS_NUM`.

> **`USAGE_NUM` is unusable.** It is zero for every row in `ARTIFACT.csv`, so
> `downstream_usage_num` has a single distinct value and carries no information. It is
> retained in the table for completeness but must not enter any model. `downstream_loc`
> (842 distinct values) and `downstream_class_num` (337) are properly populated.

In [102]:
severity = pd.read_csv(SEVERITY_CSV)[['CVE_ID', 'Severity']].drop_duplicates('CVE_ID')
cve_meta = pd.read_csv(CVE_CSV)[['CVE_ID', 'CVSS', 'CWE']].drop_duplicates('CVE_ID')

artifact = pd.read_csv(ARTIFACT_CSV)
artifact['GA'] = artifact.GROUP_ID.astype(str) + ':' + artifact.ARTIFACT_ID.astype(str)
artifact_ga = artifact.groupby('GA').agg(
    downstream_usage_num=('USAGE_NUM', 'max'),
    downstream_loc=('LOC', 'median'),
    downstream_class_num=('CLASS_NUM', 'median'),
).reset_index()

popularity = pd.read_csv(POPULARITY_CSV)[['CVE_ID', 'stars_count_at_commit']].rename(
    columns={'stars_count_at_commit': 'upstream_stars_at_fix'}
).drop_duplicates('CVE_ID')

# Blast radius per CVE. Dedupe AFTER the join: the naive merge fans out to 115,086 rows
# because DEP.csv's duplicate rows already encode CVE multiplicity.
patch = pd.read_csv(PATCH_CSV)
exposure = patch.merge(dep, left_on='G:A:V', right_on='Upstream G:A:V', how='inner')
exposure['downstream_GA'] = exposure['Downstream G:A:V'].str.rsplit(':', n=1).str[0]
exposure_pairs = exposure.drop_duplicates(['CVE_ID', 'G:A:V', 'downstream_GA'])
dependents = exposure_pairs.groupby('CVE_ID').size().rename('dependents').reset_index()

print(f'naive PATCH-DEP merge rows : {len(exposure):,}  <- do NOT use')
print(f'deduplicated exposure pairs: {len(exposure_pairs):,}  <- correct universe')

# Count from `response` (all 2,268 deduplicated mitigation records), NOT from `master`.
# `master` has already lost 591 rows to the RQ1 join, the undated filter and the pre-fix
# exclusion, so counting there would describe our dataset rather than the commit. A commit
# clearing 13 CVEs of which 9 were filtered out would otherwise be recorded as clearing 4.
commit_n = response.groupby('commit_url').CVE.nunique().rename('commit_n_CVEs')

master = (
    master
    .merge(severity, left_on='CVE', right_on='CVE_ID', how='left').drop(columns=['CVE_ID'])
    .merge(cve_meta, left_on='CVE', right_on='CVE_ID', how='left').drop(columns=['CVE_ID'])
    .merge(popularity, left_on='CVE', right_on='CVE_ID', how='left').drop(columns=['CVE_ID'])
    .merge(dependents, left_on='CVE', right_on='CVE_ID', how='left').drop(columns=['CVE_ID'])
    .merge(artifact_ga, left_on='downstream_GA', right_on='GA', how='left').drop(columns=['GA'])
    .merge(commit_n, left_on='commit_url', right_index=True, how='left')
)
master = master.rename(columns={'Severity': 'severity', 'CVSS': 'cvss', 'CWE': 'cwe'})
master['disclosure_year'] = master.disclosure_date.dt.year

cov_table = pd.DataFrame([
    {'Covariate': c, 'Non-null': int(master[c].notna().sum()),
     'Missing': int(master[c].isna().sum()),
     'Missing %': round(100 * master[c].isna().mean(), 2)}
    for c in ['severity', 'cvss', 'upstream_stars_at_fix', 'dependents',
              'downstream_usage_num', 'downstream_loc', 'commit_n_CVEs']
])
display(cov_table)

display(
    master.commit_n_CVEs.value_counts().sort_index()
    .rename_axis('CVEs fixed by the same commit').reset_index(name='Pairs').head(10)
)

naive PATCH-DEP merge rows : 115,086  <- do NOT use
deduplicated exposure pairs: 44,450  <- correct universe


,Covariate,Non-null,Missing,Missing %
0,severity,1677,0,0.00
1,cvss,1677,0,0.00
2,upstream_stars_at_fix,1655,22,1.31
3,dependents,1677,0,0.00
4,downstream_usage_num,1677,0,0.00
5,downstream_loc,1677,0,0.00
6,commit_n_CVEs,1677,0,0.00


,CVEs fixed by the same commit,Pairs
0,1,921
1,2,245
2,3,129
3,4,67
4,5,31
5,6,12
6,7,91
7,9,153
8,10,10
9,13,4


---
## T9 — Construction chain

Documents every denominator. Each row is a filtering step; the `Note` column states why
rows were removed and whether the filter originates in RQ1 or RQ2.

In [103]:
T9 = pd.DataFrame(chain)
T9['Rows removed'] = T9.Rows.shift(1) - T9.Rows
T9 = T9[['Step', 'Rows', 'Rows removed', 'Unique CVEs', 'Unique downstream repos', 'Note']]
display(T9)

expected = {'pairs': 1677, 'cves': 191, 'repos': 742}
actual = {'pairs': len(master), 'cves': master.CVE.nunique(), 'repos': master.downstream_repo.nunique()}
print('expected:', expected)
print('actual  :', actual)
print('MATCH' if actual == expected else 'MISMATCH - investigate before proceeding')

,Step,Rows,Rows removed,Unique CVEs,Unique downstream repos,Note
0,0. RESPONSE_with_commit_dates.csv (raw),3312,NaN,240,987,Starting point. One row per mined mitigation r...
1,1. drop_duplicates(),2269,1043.0,240,987,Removed byte-identical repeated rows. Not dist...
2,2. Drop downstream IDs absent from DEP.csv,2268,1.0,239,987,Malformed downstream identifiers. Cannot be jo...
3,3. Join to RQ1 lifecycle,1829,439.0,195,798,Dropped pairs whose CVE lacks a complete RQ1 l...
4,4. Drop pairs with no mined commit date,1797,32.0,195,779,Timing analysis requires an adoption date. All...
5,5. Exclude adoption before upstream fix,1677,120.0,191,742,delay_fix < 0. A commit cannot adopt a fix tha...


expected: {'pairs': 1677, 'cves': 191, 'repos': 742}
actual  : {'pairs': 1677, 'cves': 191, 'repos': 742}
MATCH


---
## T10 — Selection-bias check

**The central threat to RQ2.** `RESPONSE.csv` covers only a subset of the 593 CVEs that have
a complete RQ1 lifecycle. The CVEs it misses are *unmined*, not *unpatched*.

This table compares covered against uncovered CVEs. If they look alike, the analysis set is
arguably representative. If they differ, the paper must state which direction the bias runs.

**Already known:** coverage is balanced across classes but not across patterns. Both facts
belong in the paper — the first supports the RQ2.4 comparison, the second limits
pattern-level claims.

In [104]:
covered_ids = set(master.CVE)
cve_level = rq1_lifecycle.merge(severity, on='CVE_ID', how='left') \
                         .merge(cve_meta, on='CVE_ID', how='left') \
                         .merge(popularity, on='CVE_ID', how='left') \
                         .merge(dependents, on='CVE_ID', how='left')
cve_level['covered'] = cve_level.CVE_ID.isin(covered_ids)
cve_level['disclosure_year'] = cve_level.disclosure_date.dt.year

print(f'covered CVEs   : {cve_level.covered.sum()}')
print(f'uncovered CVEs : {(~cve_level.covered).sum()}')

rows = []
for col in ['CVSS', 'upstream_stars_at_fix', 'dependents', 'disclosure_year']:
    a = cve_level.loc[cve_level.covered, col].dropna()
    b = cve_level.loc[~cve_level.covered, col].dropna()
    if len(a) > 1 and len(b) > 1:
        _, p = mannwhitneyu(a, b)
        rows.append({'Attribute': col, 'Covered median': round(a.median(), 2),
                     'Uncovered median': round(b.median(), 2),
                     'Test': 'Mann-Whitney', 'p': round(p, 4),
                     'Balanced': 'yes' if p >= 0.05 else 'NO'})

for col in ['Severity', 'class']:
    tab = pd.crosstab(cve_level[col], cve_level.covered)
    if tab.shape[0] > 1 and tab.shape[1] > 1:
        chi2, p, _, _ = chi2_contingency(tab.values)
        rows.append({'Attribute': col, 'Covered median': '-', 'Uncovered median': '-',
                     'Test': 'Chi-square', 'p': round(p, 4),
                     'Balanced': 'yes' if p >= 0.05 else 'NO'})

T10 = pd.DataFrame(rows)
display(T10)

coverage_by_class = (
    cve_level.groupby('class').covered.agg(['sum', 'count'])
    .rename(columns={'sum': 'Covered', 'count': 'RQ1 CVEs'})
)
coverage_by_class['Coverage %'] = (100 * coverage_by_class.Covered / coverage_by_class['RQ1 CVEs']).round(1)
display(coverage_by_class)

coverage_by_pattern = (
    cve_level.groupby('pattern').covered.agg(['sum', 'count'])
    .rename(columns={'sum': 'Covered', 'count': 'RQ1 CVEs'})
)
coverage_by_pattern['Coverage %'] = (100 * coverage_by_pattern.Covered / coverage_by_pattern['RQ1 CVEs']).round(1)
display(coverage_by_pattern.sort_values('Coverage %'))

covered CVEs   : 191
uncovered CVEs : 402


,Attribute,Covered median,Uncovered median,Test,p,Balanced
0,CVSS,5.0,5.0,Mann-Whitney,0.7864,yes
1,upstream_stars_at_fix,809.0,719.5,Mann-Whitney,0.0703,yes
2,dependents,47.0,5.0,Mann-Whitney,0.0000,NO
3,disclosure_year,2018.0,2018.0,Mann-Whitney,0.3385,yes
4,Severity,-,-,Chi-square,0.4146,yes
5,class,-,-,Chi-square,0.7996,yes


,Covered,RQ1 CVEs,Coverage %
class,,,
Silent,105,332,31.6
Transparent,86,261,33.0


,Covered,RQ1 CVEs,Coverage %
pattern,,,
S2,6,40,15.0
T5,1,6,16.7
S3,4,21,19.0
T2,6,26,23.1
T3,4,13,30.8
T1,75,216,34.7
S1,95,271,35.1


---
## Validation

Three checks that must pass before any downstream notebook uses this table.

1. **Row counts** match the expected 1,797 / 195 / 779.
2. **Negative `delay_release`** should be roughly 10% of pairs — projects tracking
   snapshots can adopt before an official release tag. A very high share means a date
   column is misaligned.
3. **No duplicate keys** — the unit of analysis must be unique.

In [105]:
checks = []

checks.append({'Check': 'Pairs == 1677', 'Value': len(master), 'Pass': len(master) == 1677})
checks.append({'Check': 'CVEs == 191', 'Value': master.CVE.nunique(), 'Pass': master.CVE.nunique() == 191})
checks.append({'Check': 'Repos == 742', 'Value': master.downstream_repo.nunique(),
               'Pass': master.downstream_repo.nunique() == 742})
checks.append({'Check': 'No adoption predates the upstream fix',
               'Value': int((master.delay_fix < 0).sum()), 'Pass': (master.delay_fix < 0).sum() == 0})

dupes = master.duplicated(['CVE', 'upstream_GA', 'downstream_GA']).sum()
checks.append({'Check': 'No duplicate (CVE, upGA, downGA)', 'Value': int(dupes), 'Pass': dupes == 0})

# After the exclusion these are adoptions made after the fix but before the release tag,
# which is legitimate for projects building from source or tracking snapshots.
neg_rel = 100 * (master.delay_release < 0).mean()
checks.append({'Check': 'Negative delay_release between 1% and 15% (source/snapshot builds)',
               'Value': round(neg_rel, 1), 'Pass': 1 <= neg_rel <= 15})

n_missing_dates = master[['adoption_date', 'fix_date', 'release_date', 'disclosure_date']].isna().any(axis=1).sum()
checks.append({'Check': 'No missing core dates', 'Value': int(n_missing_dates), 'Pass': n_missing_dates == 0})

silent_with_report = master[master['class'].eq('Silent') & master.report_date.notna()]
checks.append({'Check': 'No silent row carries a report date',
               'Value': len(silent_with_report), 'Pass': len(silent_with_report) == 0})
checks.append({'Check': 'GAD rows reclassified and traceable',
               'Value': int(master.gad_reclassified.sum()), 'Pass': True})

checks_df = pd.DataFrame(checks)
display(checks_df)

if checks_df.Pass.all():
    print('ALL CHECKS PASSED')
else:
    print('FAILED:', checks_df[~checks_df.Pass].Check.tolist())

,Check,Value,Pass
0,Pairs == 1677,1677.0,True
1,CVEs == 191,191.0,True
2,Repos == 742,742.0,True
3,No adoption predates the upstream fix,0.0,True
4,"No duplicate (CVE, upGA, downGA)",0.0,True
5,Negative delay_release between 1% and 15% (sou...,3.7,True
6,No missing core dates,0.0,True
7,No silent row carries a report date,0.0,True
8,GAD rows reclassified and traceable,57.0,True


ALL CHECKS PASSED


---
## Save

Written to the `claude/` folder. Notebooks 03-06 read `rq2_master_pairwise.csv` and should
not re-derive any of the above.

In [106]:
OUT.mkdir(parents=True, exist_ok=True)

column_order = [
    'CVE', 'upstream_GA', 'downstream_GA', 'downstream_repo', 'commit_url',
    'adoption_date', 'commit_n_CVEs',
    'class', 'pattern', 'original_pattern', 'gad_reclassified', 'severity', 'cvss', 'cwe',
    'report_date', 'fix_date', 'release_date', 'disclosure_date',
    'delay_report', 'delay_fix', 'delay_release', 'delay_disclosure',
    'mitigation_stage', 'disclosure_year',
    'dependents', 'downstream_usage_num', 'downstream_loc', 'downstream_class_num',
    'upstream_stars_at_fix',
]
master_out = master[[c for c in column_order if c in master.columns]]

master_out.to_csv(OUT / 'rq2_master_pairwise.csv', index=False)
T9.to_csv(OUT / 'T9_construction_chain.csv', index=False)
T10.to_csv(OUT / 'T10_selection_bias.csv', index=False)
exposure_pairs[['CVE_ID', 'G:A:V', 'downstream_GA']].to_csv(OUT / 'rq2_exposure_universe.csv', index=False)

print(f'rq2_master_pairwise.csv    {len(master_out):>7,} rows')
print(f'rq2_exposure_universe.csv  {len(exposure_pairs):>7,} rows')
print(f'T9_construction_chain.csv  {len(T9):>7,} rows')
print(f'T10_selection_bias.csv     {len(T10):>7,} rows')
master_out.head()

rq2_master_pairwise.csv      1,677 rows
rq2_exposure_universe.csv   44,450 rows
T9_construction_chain.csv        6 rows
T10_selection_bias.csv           6 rows


,CVE,upstream_GA,downstream_GA,downstream_repo,commit_url,adoption_date,commit_n_CVEs,class,pattern,original_pattern,gad_reclassified,severity,cvss,cwe,report_date,fix_date,release_date,disclosure_date,delay_report,delay_fix,delay_release,delay_disclosure,mitigation_stage,disclosure_year,dependents,downstream_usage_num,downstream_loc,downstream_class_num,upstream_stars_at_fix
0,CVE-2021-37714,org.jsoup:jsoup,com.github.btheu.estivate:estivate,btheu/estivate,https://github.com/btheu/estivate/commit/22adf...,2021-12-03 15:44:15,1,Transparent,T1,T1,False,Medium,5.0,CWE-248;CWE-835,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,2021-08-18,110.554502,110.134062,110.003472,107.655729,After Disclosure,2021,244,0,3801.0,102.0,8521.0
1,CVE-2021-37714,org.jsoup:jsoup,com.jcabi:jcabi-http,jcabi/jcabi-http,https://github.com/jcabi/jcabi-http/commit/1a2...,2021-08-16 23:00:43,1,Transparent,T1,T1,False,Medium,5.0,CWE-248;CWE-835,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,2021-08-18,1.857604,1.437164,1.306574,-1.041169,Release->Disclosure,2021,244,0,3957.5,89.5,8521.0
2,CVE-2021-37714,org.jsoup:jsoup,in.ashwanthkumar:gocd-java-client,ashwanthkumar/gocd-java-client,https://github.com/ashwanthkumar/gocd-java-cli...,2021-09-30 23:03:50,1,Transparent,T1,T1,False,Medium,5.0,CWE-248;CWE-835,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,2021-08-18,46.859769,46.439329,46.308738,43.960995,After Disclosure,2021,244,0,1339.5,27.0,8521.0
3,CVE-2021-37714,org.jsoup:jsoup,tech.grasshopper:pdfextentreporter,grasshopper7/pdfextentreporter,https://github.com/grasshopper7/pdfextentrepor...,2022-03-01 12:31:07,1,Transparent,T1,T1,False,Medium,5.0,CWE-248;CWE-835,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,2021-08-18,198.420382,197.999942,197.869352,195.521609,After Disclosure,2021,244,0,10912.0,203.0,8521.0
4,CVE-2021-37714,org.jsoup:jsoup,de.trustable.ca3s.core:ca-3-s,kuehne-trustable-de/ca3sCore,https://github.com/kuehne-trustable-de/ca3sCor...,2021-08-23 21:32:03,1,Transparent,T1,T1,False,Medium,5.0,CWE-248;CWE-835,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,2021-08-18,8.796030,8.375590,8.245000,5.897257,After Disclosure,2021,244,0,35874.0,453.0,8521.0
